In [1]:
!pip install torch torchaudio transformers datasets scikit-learn tqdm

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 3.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 401.2 kB/s  0:30:500:00:0102:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 475.2 kB/s  0:00:25m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 435.2 kB/s  0:05:230:00:0100:08
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 461.7 kB/s  0:07:590:00:0100:13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 463.3 kB/s  0:00:01m-:--:--
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0

In [9]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Processor, Wav2Vec2Model


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET_PATH = "data"
SR = 16000
BATCH_SIZE = 8
EPOCHS = 10
MAX_LEN = 32000

In [12]:
DEVICE

device(type='cuda')

### Загрузка wav2vec

In [27]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE)
wav2vec.trainable = True
wav2vec.eval()  

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2GroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Wav2Vec2Encoder(
    (pos_conv_embed): Wav2Vec2PositionalConvEmbedding(
  

### Архитектура модели

In [17]:
class CNNRNNModel(nn.Module):
    def __init__(self, input_time=99, input_hidden=768):
        super(CNNRNNModel, self).__init__()
        
        # Вход: (Batch, 1, Time, Hidden)
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )

        # Вычисляем размер после 3-х MaxPool2d (каждый делит размер на 2)
        reduced_h = input_hidden // 8 
        
        # LSTM на вход получает (Batch, Time, Features)
        self.lstm = nn.LSTM(128 * reduced_h, 128, batch_first=True)
        self.dropout_lstm = nn.Dropout(0.3)
        
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1) # ВАЖНО: без Sigmoid для BCEWithLogitsLoss
        )

    def forward(self, x):
        # x: (Batch, Time, Hidden, 1) -> (Batch, 1, Time, Hidden)
        x = x.permute(0, 3, 1, 2)
        x = self.conv_layers(x)
        
        # Подготовка для LSTM
        batch, c, t, h = x.size()
        x = x.permute(0, 2, 1, 3).contiguous() # (Batch, Time, Channels, Hidden)
        x = x.view(batch, t, c * h)
        
        _, (hn, _) = self.lstm(x)
        x = self.dropout_lstm(hn[-1])
        return self.classifier(x)

### Подготовка датасета

In [28]:
def get_file_list():
    files, labels = [], []
    for label_val, folder in enumerate(["real", "fake"]):
        path = os.path.join(DATASET_PATH, folder)
        for f in os.listdir(path):
            files.append(os.path.join(path, f))
            labels.append(label_val)
    return files, labels

def precalculate_embeddings(files):
    all_embs = []
    print(">>> Извлечение эмбеддингов (это делается один раз)...")
    with torch.no_grad():
        for f in tqdm(files):
            audio, _ = librosa.load(f, sr=SR)
            if len(audio) > MAX_LEN: audio = audio[:MAX_LEN]
            else: audio = np.pad(audio, (0, MAX_LEN - len(audio)))
            
            inputs = processor(audio, return_tensors="pt", sampling_rate=SR).input_values.to(DEVICE)
            emb = wav2vec(inputs).last_hidden_state.squeeze(0).cpu().numpy()
            all_embs.append(emb)
    return np.array(all_embs)


files, labels = get_file_list()
X_data = precalculate_embeddings(files)
y_data = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, stratify=y_data)

class FastDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.FloatTensor(x).unsqueeze(-1) # (Time, Hidden, 1)
        self.y = torch.FloatTensor(y).unsqueeze(-1)
    def __len__(self): return len(self.x)
    def __getitem__(self, idx): return self.x[idx], self.y[idx]

train_loader = DataLoader(FastDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(FastDataset(X_test, y_test), batch_size=BATCH_SIZE)

>>> Извлечение эмбеддингов (это делается один раз)...


  0%|          | 0/14356 [00:00<?, ?it/s]

100%|██████████| 14356/14356 [01:32<00:00, 155.66it/s]


### Загрузка и обучение

In [25]:
model = CNNRNNModel().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4) # Понизили lr для стабильности


In [31]:

best_acc = 0
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Валидация
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            preds = torch.round(torch.sigmoid(model(batch_x)))
            correct += (preds == batch_y).sum().item()
    
    acc = correct / len(y_test)
    print(f"Эпоха {epoch+1}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Acc: {acc:.4f}")
    
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best_model.pth")

print(f"\nОбучение завершено! Лучшая точность: {best_acc:.4f}")

Эпоха 1/10 | Loss: 0.4984 | Acc: 0.7855
Эпоха 2/10 | Loss: 0.4912 | Acc: 0.7744
Эпоха 3/10 | Loss: 0.4876 | Acc: 0.7914
Эпоха 4/10 | Loss: 0.4810 | Acc: 0.7946
Эпоха 5/10 | Loss: 0.4797 | Acc: 0.8092
Эпоха 6/10 | Loss: 0.4730 | Acc: 0.8033
Эпоха 7/10 | Loss: 0.4677 | Acc: 0.8040
Эпоха 8/10 | Loss: 0.4640 | Acc: 0.8068
Эпоха 9/10 | Loss: 0.4611 | Acc: 0.7974
Эпоха 10/10 | Loss: 0.4647 | Acc: 0.8047

Обучение завершено! Лучшая точность: 0.8092


In [32]:
def predict_audio(file_path):
    # Загрузка и предобработка
    audio, _ = librosa.load(file_path, sr=SR)
    if len(audio) > MAX_LEN:
        audio = audio[:MAX_LEN]
    else:
        audio = np.pad(audio, (0, MAX_LEN - len(audio)))

    # Извлечение эмбеддинга
    with torch.no_grad():
        inputs = processor(audio, return_tensors="pt", sampling_rate=SR).input_values.to(DEVICE)
        emb = wav2vec(inputs).last_hidden_state # (1, Time, Hidden)
        
        # Подготовка тензора для CNN (Batch, Time, Hidden, Channel)
        emb = emb.unsqueeze(-1) 
        
        # Предсказание
        output = model(emb)
        # Применяем сигмоиду, так как в модели её нет
        probability = torch.sigmoid(output).item()
    
    label = "FAKE (Deepfake)" if probability > 0.5 else "REAL (Human)"
    print(f"Файл: {file_path}")
    print(f"Вероятность фейка: {probability:.4f}")
    print(f"Результат: {label}")
    return probability

# Пример использования:
predict_audio("luvvoice.com-20260311-y2pe3K.wav")

Файл: luvvoice.com-20260311-y2pe3K.wav
Вероятность фейка: 0.0259
Результат: REAL (Human)


0.02590807154774666